In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from sklearn.preprocessing import LabelEncoder

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# Config
# ----------------------------
columns_to_discard = [
    'collision_index',
    'collision_ref_no',
    'location_easting_osgr',
    'location_northing_osgr',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'lsoa_of_accident_location',
    'AREFNO',
    'collision_severity',
    'enhanced_severity_collision',
    'collision_injury_based',
    'collision_adjusted_severity_serious',
    'collision_adjusted_severity_slight',
    'CREFNO',
    'latitude',
    'longitude',
    'Easting',
    'Northing',
    'No. of Cas',
    'Casualty_1',
    'Casualty A',
    'junction_detail_historic',
    'junction_detail',
    'junction_control',
    'second_road_class',
    'first_road_number',
    'second_road_number',
    'Borough Na',
    '_Casualt_1',
    '_Casualt_3',
    '_Collisi_1',
    'lsoa_of_ac'
]
MAPPINGS_PATH = "categorical_mappings.json"
merged_path = 'merged_collision_data.csv'

# ----------------------------
# 1) Read merged CSV
# ----------------------------
if not os.path.exists(merged_path):
    raise FileNotFoundError(f"{merged_path} not found in current directory.")

merged_df = pd.read_csv(merged_path, low_memory=False)
print(f"Loaded merged file -> shape: {merged_df.shape}")

# Drop specified columns if present
cols_present = [c for c in columns_to_discard if c in merged_df.columns]
if cols_present:
    merged_df = merged_df.drop(columns=cols_present)
    print(f"Dropped columns_to_discard: {cols_present}")

# ----------------------------
# 2) Normalize target to 'Severity'
# ----------------------------
if 'Severity' not in merged_df.columns and 'collision_severity' in merged_df.columns:
    merged_df['Severity'] = merged_df['collision_severity']
if 'collision_severity' in merged_df.columns:
    merged_df = merged_df.drop(columns=['collision_severity'])

merged_df = merged_df.dropna(subset=['Severity'])
merged_df['Severity'] = merged_df['Severity'].astype(str).str.upper().str.strip()

print("\nTarget distribution (raw):")
print(merged_df['Severity'].value_counts())

# ----------------------------
# 3) Optional: extract hour/month if date/time exist
# ----------------------------
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], dayfirst=True, errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')
    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month
    merged_df = merged_df.drop(columns=['date', 'time'])
    print("Extracted 'hour_of_day' and 'month'")

# ----------------------------
# 4) Build X_df and detect numeric/categorical
# ----------------------------
X_df = merged_df.drop(columns=['Severity']).copy()

# Coerce obvious numeric strings to numeric
for col in X_df.columns:
    if X_df[col].dtype == 'object' or pd.api.types.is_string_dtype(X_df[col]):
        non_null = X_df[col].dropna().astype(str)
        if len(non_null) > 0:
            parsed = non_null.str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
            try:
                frac_numeric = parsed.notna().sum() / len(non_null)
            except Exception:
                frac_numeric = 0.0
            if frac_numeric > 0.9:
                X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

# Drop numeric columns containing any NaN (no numeric imputation)
numeric_cols_with_nan = [c for c in numeric_cols if X_df[c].isnull().any()]
if numeric_cols_with_nan:
    print(f"Dropping numeric columns that contain NaN (no imputations): {numeric_cols_with_nan}")
    X_df = X_df.drop(columns=numeric_cols_with_nan)
    numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"\nRemaining numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Remaining categorical columns ({len(categorical_cols)}): {categorical_cols}")

if len(numeric_cols) + len(categorical_cols) == 0:
    raise RuntimeError("No features remain after dropping numeric columns containing NaN and discarding others.")

# ----------------------------
# 5) Encode target and split (we split BEFORE creating mappings so mappings come from trainval)
# ----------------------------
# ----------------------------
# 5) Encode target as binary and split (SEVERE vs SLIGHT)
# ----------------------------
y_series = merged_df['Severity'].reset_index(drop=True)

# Binary mapping: 0 = SLIGHT, 1 = SEVERE (SERIOUS + FATAL)
y_binary = (y_series != 'SLIGHT').astype(int).values
binary_classes = np.array(['SLIGHT', 'SEVERE'])

X_trainval_df, X_test_df, y_trainval, y_test = train_test_split(
    X_df, y_binary, test_size=0.05, stratify=y_binary, random_state=SEED
)

# adjust n_splits based on smallest class in trainval
counts = np.bincount(y_trainval)
min_count = counts.min()
n_splits = 5
if min_count < n_splits:
    n_splits = int(min_count)
    if n_splits < 2:
        raise RuntimeError("Not enough samples per class to perform CV. Need at least 2 samples in smallest class.")
    print(f"Reduced StratifiedKFold n_splits to {n_splits} because smallest class has {min_count} samples.")

num_classes = 2
print(f"\nTarget classes: {list(binary_classes)} (num_classes={num_classes})")


# ----------------------------
# 6) Load saved mappings (if exists) and CREATE missing mappings from trainval, then APPLY to trainval & test
# ----------------------------
if os.path.exists(MAPPINGS_PATH):
    with open(MAPPINGS_PATH, 'r', encoding='utf-8') as fh:
        saved_mappings = json.load(fh)
        if not isinstance(saved_mappings, dict):
            saved_mappings = {}
else:
    saved_mappings = {}

# Ensure all categorical columns exist in saved_mappings; if missing, create from X_trainval_df
for col in categorical_cols:
    if col not in saved_mappings:
        print(f"Mapping for '{col}' missing. Creating mapping from train+val data and adding to {MAPPINGS_PATH}.")
        uniques = pd.Series(X_trainval_df[col].astype(str).fillna("")).unique().tolist()
        saved_mappings[col] = {str(val): idx + 1 for idx, val in enumerate(uniques)}

# Apply mapping to trainval and test (NaN or unseen -> 0)
def apply_mapping_df(df, mappings, cat_cols):
    out = df.copy()
    for col in cat_cols:
        md = mappings[col]
        out[col] = out[col].map(lambda x: int(md.get(str(x), 0)) if pd.notna(x) else 0).astype(int)
    return out

X_trainval_df = apply_mapping_df(X_trainval_df, saved_mappings, categorical_cols)
X_test_df = apply_mapping_df(X_test_df, saved_mappings, categorical_cols)

# Save updated mappings back to file
with open(MAPPINGS_PATH, 'w', encoding='utf-8') as fh:
    json.dump(saved_mappings, fh, ensure_ascii=False, indent=4)

# After mapping, categorical columns are integer-coded; feature order:
feature_names = numeric_cols + categorical_cols
print("All columns left in the data set:", feature_names)
print(f"\nAfter mappings -> numeric cols: {len(numeric_cols)}, categorical cols (coded): {len(categorical_cols)}")

# ----------------------------
# 7) Helpers for building arrays (we will use the global mappings already applied)
# ----------------------------
def build_numeric_arrays(df, numeric_cols):
    if not numeric_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    return df[numeric_cols].astype(float).to_numpy()

def build_categorical_arrays(df, categorical_cols):
    if not categorical_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    return df[categorical_cols].astype(float).to_numpy()

# ----------------------------
# 8) Model factory
# ----------------------------
def create_model(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(128, activation='relu')(inputs)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------
# 9) Stratified CV using global mappings (created from trainval)
# ----------------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

all_acc = []
all_loss = []
last_history = None
last_model = None
last_X_val = None
last_y_val = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval_df, y_trainval), start=1):
    print(f"\n--- Fold {fold} ---")
    X_train_df = X_trainval_df.iloc[train_idx].reset_index(drop=True)
    X_val_df   = X_trainval_df.iloc[val_idx].reset_index(drop=True)
    y_train = y_trainval[train_idx]
    y_val   = y_trainval[val_idx]

    # Build arrays
    X_train_num = build_numeric_arrays(X_train_df, numeric_cols)
    X_val_num = build_numeric_arrays(X_val_df, numeric_cols)

    # scale numeric per-fold
    if X_train_num.shape[1] > 0:
        means = np.nanmean(X_train_num, axis=0)
        stds = np.nanstd(X_train_num, axis=0)
        stds[stds == 0] = 1.0
        X_train_num = ((X_train_num - means) / stds).astype(np.float32)
        X_val_num = ((X_val_num - means) / stds).astype(np.float32)
    else:
        X_train_num = X_train_num.astype(np.float32)
        X_val_num = X_val_num.astype(np.float32)

    X_train_cat = build_categorical_arrays(X_train_df, categorical_cols)
    X_val_cat = build_categorical_arrays(X_val_df, categorical_cols)

    X_train = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
    X_val = np.hstack([X_val_num, X_val_cat]).astype(np.float32)

    print(f"Fold shapes -> train: {X_train.shape}, val: {X_val.shape}; classes in train: {np.unique(y_train)}")

    model = create_model(X_train.shape[1], num_classes)

    # class weights mapped to actual labels
    classes_in_fold = np.unique(y_train)
    cw = class_weight.compute_class_weight(class_weight='balanced', classes=classes_in_fold, y=y_train)
    cw_dict = {int(c): float(w) for c, w in zip(classes_in_fold, cw)}


    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        class_weight=cw_dict,
        callbacks=[es],
        verbose=1
    )

    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold} -> val_loss: {loss:.4f}, val_acc: {acc:.4f}")
    all_acc.append(acc)
    all_loss.append(loss)

    last_history = history
    last_model = model
    last_X_val = X_val
    last_y_val = y_val

# ----------------------------
# 10) CV summary + plots
# ----------------------------
print(f"\nAverage Validation Accuracy: {np.mean(all_acc):.4f} (+/- {np.std(all_acc):.4f})")
print(f"Average Validation Loss: {np.mean(all_loss):.4f} (+/- {np.std(all_loss):.4f})")

if last_history is not None:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(last_history.history.get('accuracy', []), label='train_acc')
    plt.plot(last_history.history.get('val_accuracy', []), label='val_acc')
    plt.legend(); plt.title('Last fold acc')
    plt.subplot(1,2,2)
    plt.plot(last_history.history.get('loss', []), label='train_loss')
    plt.plot(last_history.history.get('val_loss', []), label='val_loss')
    plt.legend(); plt.title('Last fold loss')
    plt.tight_layout(); plt.show()

if last_model is not None and last_X_val is not None:
    y_pred_prob = last_model.predict(last_X_val)
    y_pred = np.argmax(y_pred_prob, axis=1)

    unique_labels_in_fold = np.unique(last_y_val)
    target_names_in_fold = binary_classes[unique_labels_in_fold]


    print("\nClassification report (last fold):")
    print(classification_report(last_y_val, y_pred, labels=unique_labels_in_fold, target_names=target_names_in_fold))

    cm = confusion_matrix(last_y_val, y_pred)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names_in_fold, yticklabels=target_names_in_fold)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (last fold)'); plt.show()

    # Permutation importance (last fold)
    print("\nRunning permutation importance (this may take a bit)...")
    baseline_recall = recall_score(last_y_val, y_pred, labels=unique_labels_in_fold, average=None)
    n_repeats = 10
    rng = np.random.RandomState(42)

    feat_importance = np.zeros((len(feature_names), len(unique_labels_in_fold)), dtype=float)

    for fi in range(len(feature_names)):
        drops = np.zeros((n_repeats, len(unique_labels_in_fold)), dtype=float)
        col = last_X_val[:, fi].copy()
        for r in range(n_repeats):
            perm = col.copy()
            rng.shuffle(perm)
            X_perm = last_X_val.copy()
            X_perm[:, fi] = perm
            y_perm_prob = last_model.predict(X_perm, verbose=0)
            y_perm = np.argmax(y_perm_prob, axis=1)
            perm_recall = recall_score(last_y_val, y_perm, labels=unique_labels_in_fold, average=None)
            drops[r, :] = baseline_recall - perm_recall
        feat_importance[fi, :] = drops.mean(axis=0)

    top_k = 10
    for ci, lbl in enumerate(unique_labels_in_fold):
        cls_name = binary_classes[int(lbl)]
        importance_for_class = feat_importance[:, ci]
        sorted_idx = np.argsort(-importance_for_class)
        top_idx = sorted_idx[:top_k]
        print(f"\nTop {min(top_k, len(feature_names))} features for class '{cls_name}' (mean recall drop):")
        for rank, idx in enumerate(top_idx, start=1):
            fname = feature_names[idx]
            val = importance_for_class[idx]
            print(f" {rank:2d}. {fname:40s}  drop={val:.4f}")

        plt.figure(figsize=(8,4))
        names = [feature_names[i] for i in top_idx]
        vals = importance_for_class[top_idx]
        plt.barh(range(len(vals))[::-1], vals)
        plt.yticks(range(len(vals))[::-1], names)
        plt.xlabel('Mean drop in recall after permutation')
        plt.title(f"Top features for class '{cls_name}'")
        plt.tight_layout()
        plt.show()

# ----------------------------
# 11) Final model trained on full trainval and evaluated on test (no leakage)
# ----------------------------
# Build final trainval and test arrays using the SAME mappings (already applied)
# Numeric scaling using full trainval
X_trainval_num = build_numeric_arrays(X_trainval_df, numeric_cols)
X_test_num = build_numeric_arrays(X_test_df, numeric_cols)

if X_trainval_num.shape[1] > 0:
    final_means = np.nanmean(X_trainval_num, axis=0)
    final_stds = np.nanstd(X_trainval_num, axis=0)
    final_stds[final_stds == 0] = 1.0
    X_trainval_num = ((X_trainval_num - final_means) / final_stds).astype(np.float32)
    X_test_num = ((X_test_num - final_means) / final_stds).astype(np.float32)

X_trainval_cat = build_categorical_arrays(X_trainval_df, categorical_cols)
X_test_cat = build_categorical_arrays(X_test_df, categorical_cols)

X_trainval_final = np.hstack([X_trainval_num, X_trainval_cat]).astype(np.float32)
X_test_final = np.hstack([X_test_num, X_test_cat]).astype(np.float32)

classes_full = np.unique(y_trainval)
cw_full = class_weight.compute_class_weight(class_weight='balanced', classes=classes_full, y=y_trainval)
cw_full_dict = {int(c): float(w) for c, w in zip(classes_full, cw_full)}

final_model = create_model(X_trainval_final.shape[1], num_classes)
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
final_model.fit(X_trainval_final, y_trainval, validation_split=0.1, epochs=100, batch_size=64,
                class_weight=cw_full_dict, callbacks=[es], verbose=1)

y_test_pred = np.argmax(final_model.predict(X_test_final), axis=1)

print("\n--- FINAL TEST SET EVALUATION (UNSEEN DATA) ---")
print(classification_report(y_test, y_test_pred, target_names=binary_classes))



In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ----------------------------
# Config
# ----------------------------
columns_to_discard = [
    'collision_index', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr',
    'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current',
    'lsoa_of_accident_location', 'AREFNO', 'collision_severity', 'enhanced_severity_collision',
    'collision_injury_based', 'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight',
    'CREFNO', 'latitude', 'longitude', 'Easting', 'Northing', 'No. of Cas', 'Casualty_1',
    'Casualty A', 'junction_detail_historic', 'junction_detail', 'junction_control',
    'second_road_class', 'first_road_number', 'second_road_number', 'Borough Na',
    '_Casualt_1', '_Casualt_3', '_Collisi_1', 'lsoa_of_ac'
]
MAPPINGS_PATH = "categorical_mappings.json"
merged_path = 'merged_collision_data.csv'

# ----------------------------
# 1) Read merged CSV
# ----------------------------
merged_df = pd.read_csv(merged_path, low_memory=False)
cols_present = [c for c in columns_to_discard if c in merged_df.columns]
if cols_present:
    merged_df = merged_df.drop(columns=cols_present)

# ----------------------------
# 2) Target column
# ----------------------------
merged_df = merged_df.dropna(subset=['Severity'])
merged_df['Severity'] = merged_df['Severity'].astype(str).str.upper().str.strip()

# Binary target: 0 = SLIGHT, 1 = SEVERE
y_series = merged_df['Severity']
y_binary = (y_series != 'SLIGHT').astype(int).values
binary_classes = np.array(['SLIGHT', 'SEVERE'])

# ----------------------------
# 3) Extract hour/month if date/time exist
# ----------------------------
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], dayfirst=True, errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')
    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month
    merged_df = merged_df.drop(columns=['date', 'time'])

# ----------------------------
# 4) Build features
# ----------------------------
X_df = merged_df.drop(columns=['Severity']).copy()

# Convert numeric-like strings
for col in X_df.columns:
    if X_df[col].dtype == 'object' or pd.api.types.is_string_dtype(X_df[col]):
        non_null = X_df[col].dropna().astype(str)
        if len(non_null) > 0:
            parsed = non_null.str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
            frac_numeric = parsed.notna().sum() / len(non_null)
            if frac_numeric > 0.9:
                X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

# Drop numeric columns with NaN
numeric_cols_with_nan = [c for c in numeric_cols if X_df[c].isnull().any()]
if numeric_cols_with_nan:
    X_df = X_df.drop(columns=numeric_cols_with_nan)
    numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"Numeric cols: {numeric_cols}")
print(f"Categorical cols: {categorical_cols}")

# ----------------------------
# 5) Train/test split
# ----------------------------
X_trainval_df, X_test_df, y_trainval, y_test = train_test_split(
    X_df, y_binary, test_size=0.1, stratify=y_binary, random_state=SEED
)

# ----------------------------
# 6) Encode categorical columns
# ----------------------------
if os.path.exists(MAPPINGS_PATH):
    with open(MAPPINGS_PATH, 'r', encoding='utf-8') as fh:
        saved_mappings = json.load(fh)
else:
    saved_mappings = {}

for col in categorical_cols:
    if col not in saved_mappings:
        uniques = pd.Series(X_trainval_df[col].astype(str).fillna("")).unique().tolist()
        saved_mappings[col] = {str(val): idx + 1 for idx, val in enumerate(uniques)}

def apply_mapping_df(df, mappings, cat_cols):
    out = df.copy()
    for col in cat_cols:
        md = mappings[col]
        out[col] = out[col].map(lambda x: int(md.get(str(x), 0)) if pd.notna(x) else 0).astype(int)
    return out

X_trainval_df = apply_mapping_df(X_trainval_df, saved_mappings, categorical_cols)
X_test_df = apply_mapping_df(X_test_df, saved_mappings, categorical_cols)

with open(MAPPINGS_PATH, 'w', encoding='utf-8') as fh:
    json.dump(saved_mappings, fh, ensure_ascii=False, indent=4)

# ----------------------------
# 7) Feature engineering: interactions
# ----------------------------
X_trainval_df['vehicles_x_speed'] = X_trainval_df['number_of_vehicles'] * X_trainval_df['speed_limit']
X_trainval_df['casualties_per_vehicle'] = X_trainval_df['number_of_casualties'] / (X_trainval_df['number_of_vehicles'] + 1)
X_trainval_df['hour_urban'] = X_trainval_df['hour_of_day'] * X_trainval_df['urban_or_rural_area']
X_trainval_df['weather_road'] = X_trainval_df['weather_conditions'].astype(str) + "_" + X_trainval_df['road_surface_conditions'].astype(str)

X_test_df['vehicles_x_speed'] = X_test_df['number_of_vehicles'] * X_test_df['speed_limit']
X_test_df['casualties_per_vehicle'] = X_test_df['number_of_casualties'] / (X_test_df['number_of_vehicles'] + 1)
X_test_df['hour_urban'] = X_test_df['hour_of_day'] * X_test_df['urban_or_rural_area']
X_test_df['weather_road'] = X_test_df['weather_conditions'].astype(str) + "_" + X_test_df['road_surface_conditions'].astype(str)

# Add new weather_road as categorical mapping
for col in ['weather_road']:
    uniques = pd.Series(X_trainval_df[col].astype(str)).unique().tolist()
    mapping = {val: idx+1 for idx, val in enumerate(uniques)}
    X_trainval_df[col] = X_trainval_df[col].map(lambda x: mapping.get(str(x), 0)).astype(int)
    X_test_df[col] = X_test_df[col].map(lambda x: mapping.get(str(x), 0)).astype(int)
    categorical_cols.append(col)

# ----------------------------
# 8) Convert to numpy arrays
# ----------------------------
X_trainval = np.hstack([X_trainval_df[numeric_cols].values.astype(float),
                        X_trainval_df[categorical_cols].values.astype(int)])
X_test = np.hstack([X_test_df[numeric_cols].values.astype(float),
                    X_test_df[categorical_cols].values.astype(int)])

# Fill NaNs with median
for i, col in enumerate(numeric_cols):
    median = np.nanmedian(X_trainval[:, i])
    X_trainval[:, i] = np.where(np.isnan(X_trainval[:, i]), median, X_trainval[:, i])
    X_test[:, i] = np.where(np.isnan(X_test[:, i]), median, X_test[:, i])

# ----------------------------
# 9) Oversample SEVERE using SMOTE
# ----------------------------
smote = SMOTE(random_state=SEED)
X_trainval_res, y_trainval_res = smote.fit_resample(X_trainval, y_trainval)
print("After SMOTE, class counts:", np.bincount(y_trainval_res))

# ----------------------------
# 10) Train LightGBM
# ----------------------------
lgb_train = lgb.Dataset(X_trainval_res, label=y_trainval_res)
lgb_valid = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'seed': SEED,
    'verbose': -1
}

callbacks = [
    lgb.early_stopping(stopping_rounds=20),
    lgb.log_evaluation(period=50)
]

gbm = lgb.train(params, lgb_train, num_boost_round=500, valid_sets=[lgb_train, lgb_valid], callbacks=callbacks)

# ----------------------------
# 11) Threshold tuning for SEVERE
# ----------------------------
y_val_probs = gbm.predict(X_test)
best_thresh = 0.5
best_f1 = 0
for thresh in np.arange(0.05, 0.5, 0.05):
    y_pred_thresh = (y_val_probs > thresh).astype(int)
    f1 = f1_score(y_test, y_pred_thresh)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh
print("Optimal threshold for SEVERE:", best_thresh)

y_pred_final = (y_val_probs > best_thresh).astype(int)

# ----------------------------
# 12) Evaluation
# ----------------------------
print("\n--- TEST SET EVALUATION ---")
print(classification_report(y_test, y_pred_final, target_names=binary_classes))

cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=binary_classes, yticklabels=binary_classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, recall_score

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# Config
# ----------------------------
columns_to_discard = [
    'collision_index','collision_ref_no','location_easting_osgr','location_northing_osgr',
    'local_authority_ons_district','local_authority_highway','local_authority_highway_current',
    'lsoa_of_accident_location','AREFNO','collision_severity','enhanced_severity_collision',
    'collision_injury_based','collision_adjusted_severity_serious','collision_adjusted_severity_slight',
    'CREFNO','latitude','longitude','Easting','Northing','No. of Cas','Casualty_1','Casualty A',
    'junction_detail_historic','junction_detail','junction_control','second_road_class',
    'first_road_number','second_road_number','Borough Na','_Casualt_1','_Casualt_3','_Collisi_1','lsoa_of_ac'
]
MAPPINGS_PATH = "categorical_mappings.json"
merged_path = 'merged_collision_data.csv'

# ----------------------------
# 1) Load data
# ----------------------------
merged_df = pd.read_csv(merged_path, low_memory=False)
cols_present = [c for c in columns_to_discard if c in merged_df.columns]
merged_df = merged_df.drop(columns=cols_present)
print(f"Loaded merged file -> shape: {merged_df.shape}")

# ----------------------------
# 2) Target processing
# ----------------------------
merged_df = merged_df.dropna(subset=['Severity'])
merged_df['Severity'] = merged_df['Severity'].astype(str).str.upper().str.strip()
y_series = merged_df['Severity']
print("\nTarget distribution (raw):")
print(y_series.value_counts())

# Binary: SLIGHT=0, SEVERE=1
y_binary = (y_series != 'SLIGHT').astype(int).values
binary_classes = np.array(['SLIGHT','SEVERE'])

# ----------------------------
# 3) Optional: hour/month extraction
# ----------------------------
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], dayfirst=True, errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')
    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month
    merged_df = merged_df.drop(columns=['date','time'])
    print("Extracted 'hour_of_day' and 'month'")

# ----------------------------
# 4) Features: numeric / categorical
# ----------------------------
X_df = merged_df.drop(columns=['Severity']).copy()

# Coerce numeric-like columns
for col in X_df.columns:
    if X_df[col].dtype == 'object':
        X_df[col] = pd.to_numeric(X_df[col].astype(str).str.replace(r'[^\d\.\-]', '', regex=True), errors='coerce')

numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

# Drop numeric columns with NaN
numeric_cols_with_nan = [c for c in numeric_cols if X_df[c].isnull().any()]
X_df = X_df.drop(columns=numeric_cols_with_nan)
numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"Numeric cols ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical cols ({len(categorical_cols)}): {categorical_cols}")

# ----------------------------
# 5) Train/test split
# ----------------------------
X_trainval_df, X_test_df, y_trainval, y_test = train_test_split(
    X_df, y_binary, test_size=0.05, stratify=y_binary, random_state=SEED
)

# ----------------------------
# 6) Encode categorical features as integer indices for embeddings
# ----------------------------
if os.path.exists(MAPPINGS_PATH):
    with open(MAPPINGS_PATH, 'r') as fh:
        saved_mappings = json.load(fh)
else:
    saved_mappings = {}

for col in categorical_cols:
    if col not in saved_mappings:
        uniques = pd.Series(X_trainval_df[col].astype(str).fillna("")).unique().tolist()
        saved_mappings[col] = {str(val): idx+1 for idx,val in enumerate(uniques)}

def apply_mapping(df, mappings, cat_cols):
    out = df.copy()
    for col in cat_cols:
        md = mappings[col]
        out[col] = out[col].map(lambda x: int(md.get(str(x),0))).astype(int)
    return out

X_trainval_df = apply_mapping(X_trainval_df, saved_mappings, categorical_cols)
X_test_df = apply_mapping(X_test_df, saved_mappings, categorical_cols)

with open(MAPPINGS_PATH,'w') as fh:
    json.dump(saved_mappings,fh,indent=2)

# ----------------------------
# 7) Scale numeric features
# ----------------------------
X_trainval_num = X_trainval_df[numeric_cols].astype(float).to_numpy()
X_test_num = X_test_df[numeric_cols].astype(float).to_numpy()

if X_trainval_num.shape[1]>0:
    means = np.nanmean(X_trainval_num, axis=0)
    stds = np.nanstd(X_trainval_num, axis=0)
    stds[stds==0] = 1
    X_trainval_num = ((X_trainval_num - means)/stds).astype(np.float32)
    X_test_num = ((X_test_num - means)/stds).astype(np.float32)

X_trainval_cat = X_trainval_df[categorical_cols].to_numpy()
X_test_cat = X_test_df[categorical_cols].to_numpy()

# ----------------------------
# 8) Neural network with embedding layers
# ----------------------------
def create_nn_model(n_numeric, cat_sizes, emb_dims):
    numeric_input = keras.Input(shape=(n_numeric,), name='numeric')
    cat_inputs = []
    emb_layers = []
    for i, (cats, dim) in enumerate(zip(cat_sizes, emb_dims)):
        inp = keras.Input(shape=(1,), name=f'cat_{i}')
        emb = keras.layers.Embedding(input_dim=cats+1, output_dim=dim, input_length=1)(inp)
        emb = keras.layers.Reshape(target_shape=(dim,))(emb)
        cat_inputs.append(inp)
        emb_layers.append(emb)
    x = keras.layers.Concatenate()([numeric_input]+emb_layers)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(2, activation='softmax')(x)
    model = keras.Model(inputs=[numeric_input]+cat_inputs, outputs=out)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Embedding dimensions (rule of thumb: min(50, number of categories//2))
cat_sizes = [len(saved_mappings[col]) for col in categorical_cols]
emb_dims = [min(50, max(1, size//2)) for size in cat_sizes]

# ----------------------------
# 9) Stratified K-Fold CV with SMOTE per fold
# ----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
all_acc = []
all_loss = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval_num, y_trainval), start=1):
    print(f"\n--- Fold {fold} ---")
    X_tr_num, X_val_num = X_trainval_num[train_idx], X_trainval_num[val_idx]
    X_tr_cat, X_val_cat = X_trainval_cat[train_idx], X_trainval_cat[val_idx]
    y_tr, y_val = y_trainval[train_idx], y_trainval[val_idx]

    # ----------------------------
    # SMOTE oversampling for numeric+categorical combined
    # ----------------------------
    X_tr_comb = np.hstack([X_tr_num, X_tr_cat])
    sm = SMOTE(random_state=SEED)
    X_tr_res, y_tr_res = sm.fit_resample(X_tr_comb, y_tr)

    # Split back into numeric + categorical
    X_tr_num_res = X_tr_res[:, :X_tr_num.shape[1]].astype(np.float32)
    X_tr_cat_res = X_tr_res[:, X_tr_num.shape[1]:].astype(int)

    # ----------------------------
    # Class weights
    # ----------------------------
    classes_in_fold = np.unique(y_tr_res)
    cw = class_weight.compute_class_weight(class_weight='balanced', classes=classes_in_fold, y=y_tr_res)
    cw_dict = {int(c): float(w) for c, w in zip(classes_in_fold, cw)}

    # ----------------------------
    # Build model
    # ----------------------------
    model = create_nn_model(n_numeric=X_tr_num_res.shape[1], cat_sizes=cat_sizes, emb_dims=emb_dims)

    # ----------------------------
    # Fit model
    # ----------------------------
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(
        [X_tr_num_res]+[X_tr_cat_res[:,i] for i in range(X_tr_cat_res.shape[1])],
        y_tr_res,
        validation_data=([X_val_num]+[X_val_cat[:,i] for i in range(X_val_cat.shape[1])], y_val),
        epochs=50,
        batch_size=64,
        class_weight=cw_dict,
        callbacks=[es],
        verbose=1
    )

    loss, acc = model.evaluate([X_val_num]+[X_val_cat[:,i] for i in range(X_val_cat.shape[1])], y_val, verbose=0)
    print(f"Fold {fold} -> val_loss: {loss:.4f}, val_acc: {acc:.4f}")
    all_acc.append(acc)
    all_loss.append(loss)

print(f"\nAverage Validation Accuracy: {np.mean(all_acc):.4f} (+/- {np.std(all_acc):.4f})")

# ----------------------------
# 10) Final model on full trainval
# ----------------------------
# SMOTE on full trainval
X_trainval_comb = np.hstack([X_trainval_num, X_trainval_cat])
sm = SMOTE(random_state=SEED)
X_trainval_res, y_trainval_res = sm.fit_resample(X_trainval_comb, y_trainval)
X_trainval_num_res = X_trainval_res[:, :X_trainval_num.shape[1]].astype(np.float32)
X_trainval_cat_res = X_trainval_res[:, X_trainval_num.shape[1]:].astype(int)

# Class weights
classes_full = np.unique(y_trainval_res)
cw_full = class_weight.compute_class_weight(class_weight='balanced', classes=classes_full, y=y_trainval_res)
cw_full_dict = {int(c): float(w) for c, w in zip(classes_full, cw_full)}

# Build and train
final_model = create_nn_model(n_numeric=X_trainval_num_res.shape[1], cat_sizes=cat_sizes, emb_dims=emb_dims)
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
final_model.fit(
    [X_trainval_num_res]+[X_trainval_cat_res[:,i] for i in range(X_trainval_cat_res.shape[1])],
    y_trainval_res,
    validation_split=0.1,
    epochs=100,
    batch_size=64,
    class_weight=cw_full_dict,
    callbacks=[es],
    verbose=1
)

# ----------------------------
# 11) Evaluate on test
# ----------------------------
y_test_pred_prob = final_model.predict([X_test_num]+[X_test_cat[:,i] for i in range(X_test_cat.shape[1])])
y_test_pred = np.argmax(y_test_pred_prob, axis=1)

print("\n--- FINAL TEST SET EVALUATION (UNSEEN DATA) ---")
print(classification_report(y_test, y_test_pred, target_names=binary_classes))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=binary_classes, yticklabels=binary_classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.impute import SimpleImputer
from tensorflow import keras
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from sklearn.preprocessing import LabelEncoder

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ----------------------------
# Config
# ----------------------------
columns_to_discard = [
    'collision_index', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr',
    'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current',
    'lsoa_of_accident_location', 'AREFNO', 'collision_severity', 'enhanced_severity_collision',
    'collision_injury_based', 'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight',
    'CREFNO', 'latitude', 'longitude', 'Easting', 'Northing', 'No. of Cas',
    'Casualty_1', 'Casualty A', 'junction_detail_historic', 'junction_detail', 'junction_control',
    'second_road_class', 'first_road_number', 'second_road_number', 'Borough Na',
    '_Casualt_1', '_Casualt_3', '_Collisi_1', 'lsoa_of_ac'
]
MAPPINGS_PATH = "categorical_mappings.json"
merged_path = 'merged_collision_data.csv'

# ----------------------------
# 1) Read merged CSV
# ----------------------------
if not os.path.exists(merged_path):
    raise FileNotFoundError(f"{merged_path} not found in current directory.")

merged_df = pd.read_csv(merged_path, low_memory=False)
print(f"Loaded merged file -> shape: {merged_df.shape}")

# Drop specified columns if present
cols_present = [c for c in columns_to_discard if c in merged_df.columns]
if cols_present:
    merged_df = merged_df.drop(columns=cols_present)
    print(f"Dropped columns_to_discard: {cols_present}")

# ----------------------------
# 2) Normalize target to 'Severity'
# ----------------------------
if 'Severity' not in merged_df.columns and 'collision_severity' in merged_df.columns:
    merged_df['Severity'] = merged_df['collision_severity']
if 'collision_severity' in merged_df.columns:
    merged_df = merged_df.drop(columns=['collision_severity'])

merged_df = merged_df.dropna(subset=['Severity'])
merged_df['Severity'] = merged_df['Severity'].astype(str).str.upper().str.strip()

print("\nTarget distribution (raw):")
print(merged_df['Severity'].value_counts())

# ----------------------------
# 3) Optional: extract hour/month if date/time exist
# ----------------------------
if 'date' in merged_df.columns and 'time' in merged_df.columns:
    merged_df['date'] = pd.to_datetime(merged_df['date'], dayfirst=True, errors='coerce')
    merged_df['time'] = pd.to_datetime(merged_df['time'], format='%H:%M:%S', errors='coerce')
    merged_df['hour_of_day'] = merged_df['time'].dt.hour
    merged_df['month'] = merged_df['date'].dt.month
    merged_df = merged_df.drop(columns=['date', 'time'])
    print("Extracted 'hour_of_day' and 'month'")

# ----------------------------
# 4) Build X_df and detect numeric/categorical (CLEANUP ONLY)
# ----------------------------
X_df = merged_df.drop(columns=['Severity']).copy()

# Parse obvious numeric strings (No leakage here, just type conversion)
for col in X_df.columns:
    if X_df[col].dtype == 'object' or pd.api.types.is_string_dtype(X_df[col]):
        non_null = X_df[col].dropna().astype(str)
        if len(non_null) > 0:
            parsed = non_null.str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
            try:
                frac_numeric = parsed.notna().sum() / len(non_null)
            except Exception:
                frac_numeric = 0.0
            if frac_numeric > 0.9:
                X_df[col] = pd.to_numeric(X_df[col], errors='coerce')

numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(exclude=['number']).columns.tolist()

print(f"\nInitial numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Initial categorical columns ({len(categorical_cols)}): {categorical_cols}")

# ----------------------------
# 5) Encode target and split (SPLIT BEFORE IMPUTATION)
# ----------------------------
y_series = merged_df['Severity'].reset_index(drop=True)
y_binary = (y_series != 'SLIGHT').astype(int).values
binary_classes = np.array(['SLIGHT', 'SEVERE'])

X_trainval_df, X_test_df, y_trainval, y_test = train_test_split(
    X_df, y_binary, test_size=0.05, stratify=y_binary, random_state=SEED
)

# ----------------------------
# 6) Identify Valid Numeric Columns (Based on TrainVal)
# ----------------------------
# If a column is ALL NaN in the training set, we cannot impute it (median is undefined).
# We must drop these columns from both TrainVal and Test to avoid shape errors.
valid_numeric_cols = [c for c in numeric_cols if X_trainval_df[c].notna().any()]
dropped_numeric = list(set(numeric_cols) - set(valid_numeric_cols))

if dropped_numeric:
    print(f"Dropping all-NaN numeric columns: {dropped_numeric}")
    numeric_cols = valid_numeric_cols
    # We don't drop from DF here, we just won't use them in the 'numeric_cols' list for imputing/scaling

# ----------------------------
# 7) Mappings (Created from TrainVal)
# ----------------------------
if os.path.exists(MAPPINGS_PATH):
    with open(MAPPINGS_PATH, 'r', encoding='utf-8') as fh:
        saved_mappings = json.load(fh)
        if not isinstance(saved_mappings, dict):
            saved_mappings = {}
else:
    saved_mappings = {}

# Create missing mappings from X_trainval_df
for col in categorical_cols:
    if col not in saved_mappings:
        print(f"Creating mapping for '{col}' from train+val data.")
        uniques = pd.Series(X_trainval_df[col].astype(str).fillna("")).unique().tolist()
        saved_mappings[col] = {str(val): idx + 1 for idx, val in enumerate(uniques)}

# Apply mapping
def apply_mapping_df(df, mappings, cat_cols):
    out = df.copy()
    for col in cat_cols:
        md = mappings[col]
        out[col] = out[col].map(lambda x: int(md.get(str(x), 0)) if pd.notna(x) else 0).astype(int)
    return out

X_trainval_df = apply_mapping_df(X_trainval_df, saved_mappings, categorical_cols)
X_test_df = apply_mapping_df(X_test_df, saved_mappings, categorical_cols)

# Save mappings
with open(MAPPINGS_PATH, 'w', encoding='utf-8') as fh:
    json.dump(saved_mappings, fh, ensure_ascii=False, indent=4)

feature_names = numeric_cols + categorical_cols

# ----------------------------
# 8) Helpers
# ----------------------------
def build_numeric_arrays(df, num_cols):
    if not num_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    return df[num_cols].astype(float).to_numpy()

def build_categorical_arrays(df, cat_cols):
    if not cat_cols:
        return np.zeros((len(df), 0), dtype=np.float32)
    return df[cat_cols].astype(float).to_numpy()

def create_model(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(128, activation='relu')(inputs)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------
# 9) Stratified CV (Imputation Inside Loop)
# ----------------------------
# Adjust splits
counts = np.bincount(y_trainval)
n_splits = 5 if counts.min() >= 5 else int(counts.min())
if n_splits < 2: raise RuntimeError("Not enough samples for CV")

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

all_acc = []
all_loss = []
last_history = None
last_model = None
last_X_val = None
last_y_val = None

print("\nStarting Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval_df, y_trainval), start=1):
    print(f"\n--- Fold {fold} ---")

    # Use .copy() to avoid SettingWithCopyWarning when we impute
    X_train_fold = X_trainval_df.iloc[train_idx].copy()
    X_val_fold   = X_trainval_df.iloc[val_idx].copy()
    y_train = y_trainval[train_idx]
    y_val   = y_trainval[val_idx]

    # --- IMPUTATION (Fit on Fold Train, Transform Fold Val) ---
    if numeric_cols:
        # Safety check: only impute columns that have data in this specific training fold
        # to avoid "Columns must be same length as key" error
        fold_valid_nums = [c for c in numeric_cols if X_train_fold[c].notna().any()]

        if fold_valid_nums:
            imputer_cv = SimpleImputer(strategy='median')
            X_train_fold[fold_valid_nums] = imputer_cv.fit_transform(X_train_fold[fold_valid_nums])
            X_val_fold[fold_valid_nums]   = imputer_cv.transform(X_val_fold[fold_valid_nums])

    # Build arrays
    X_train_num = build_numeric_arrays(X_train_fold, numeric_cols)
    X_val_num   = build_numeric_arrays(X_val_fold, numeric_cols)

    # Scale numeric per-fold
    if X_train_num.shape[1] > 0:
        means = np.nanmean(X_train_num, axis=0)
        stds = np.nanstd(X_train_num, axis=0)
        stds[stds == 0] = 1.0
        X_train_num = ((X_train_num - means) / stds).astype(np.float32)
        X_val_num   = ((X_val_num - means) / stds).astype(np.float32)

    X_train_cat = build_categorical_arrays(X_train_fold, categorical_cols)
    X_val_cat   = build_categorical_arrays(X_val_fold, categorical_cols)

    X_train = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
    X_val   = np.hstack([X_val_num, X_val_cat]).astype(np.float32)

    # Train
    classes_in_fold = np.unique(y_train)
    cw = class_weight.compute_class_weight(class_weight='balanced', classes=classes_in_fold, y=y_train)
    cw_dict = {int(c): float(w) for c, w in zip(classes_in_fold, cw)}

    model = create_model(X_train.shape[1], num_classes)
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        class_weight=cw_dict,
        callbacks=[es],
        verbose=0 # Reduced verbosity for CV
    )

    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold} -> val_loss: {loss:.4f}, val_acc: {acc:.4f}")
    all_acc.append(acc)
    all_loss.append(loss)

    last_history = history
    last_model = model
    last_X_val = X_val
    last_y_val = y_val

print(f"\nAverage Validation Accuracy: {np.mean(all_acc):.4f}")

# ----------------------------
# 10) Final Model (TrainVal -> Test)
# ----------------------------
# Prepare full TrainVal and Test sets
X_trainval_final_df = X_trainval_df.copy()
X_test_final_df     = X_test_df.copy()

# --- IMPUTATION (Fit on TrainVal, Transform Test) ---
if numeric_cols:
    imputer_final = SimpleImputer(strategy='median')
    X_trainval_final_df[numeric_cols] = imputer_final.fit_transform(X_trainval_final_df[numeric_cols])
    X_test_final_df[numeric_cols]     = imputer_final.transform(X_test_final_df[numeric_cols])

# Build arrays
X_trainval_num = build_numeric_arrays(X_trainval_final_df, numeric_cols)
X_test_num     = build_numeric_arrays(X_test_final_df, numeric_cols)

# Scale using TrainVal stats
if X_trainval_num.shape[1] > 0:
    final_means = np.nanmean(X_trainval_num, axis=0)
    final_stds  = np.nanstd(X_trainval_num, axis=0)
    final_stds[final_stds == 0] = 1.0
    X_trainval_num = ((X_trainval_num - final_means) / final_stds).astype(np.float32)
    X_test_num     = ((X_test_num - final_means) / final_stds).astype(np.float32)

X_trainval_cat = build_categorical_arrays(X_trainval_final_df, categorical_cols)
X_test_cat     = build_categorical_arrays(X_test_final_df, categorical_cols)

X_trainval_final = np.hstack([X_trainval_num, X_trainval_cat]).astype(np.float32)
X_test_final     = np.hstack([X_test_num, X_test_cat]).astype(np.float32)

# Train Final
print("\nTraining Final Model on full TrainVal set...")
classes_full = np.unique(y_trainval)
cw_full = class_weight.compute_class_weight(class_weight='balanced', classes=classes_full, y=y_trainval)
cw_full_dict = {int(c): float(w) for c, w in zip(classes_full, cw_full)}

final_model = create_model(X_trainval_final.shape[1], num_classes)
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
final_model.fit(
    X_trainval_final, y_trainval,
    validation_split=0.1,
    epochs=100,
    batch_size=64,
    class_weight=cw_full_dict,
    callbacks=[es],
    verbose=1
)

y_test_pred = np.argmax(final_model.predict(X_test_final), axis=1)

print("\n--- FINAL TEST SET EVALUATION (UNSEEN DATA) ---")
print(classification_report(y_test, y_test_pred, target_names=binary_classes))

cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['SLIGHT', 'SEVERE'],
    yticklabels=['SLIGHT', 'SEVERE']
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix – Unseen Test Set')
plt.tight_layout()
plt.show()
